## Specify the Knowledge Base to use

In [1]:
from importlib.resources import files
import kmds
from kmds.tagging.tag_types import DataRepresentationTags
from owlready2 import *
from kmds.utils.load_utils import *
from kmds.utils.path_utils import get_package_kb_path
KNOWLEDGE_BASE = str(files("kmds.examples").joinpath("example_analytics_kb_app_workflow.xml"))
onto2 :Ontology = load_kb(KNOWLEDGE_BASE)
import pandas as pd
pd.set_option('display.max_colwidth', None)

## Load the Exploratory Observations

In [2]:

load_exp_observations(onto2)

,obs_type,intent,finding,finding_seq
0,Data Quality Observation,DATA_UNDERSTANDING,This use case computes the time to resolve a ticket. This is time that has elapsed between the ticket creation and the ticketclosed time stamps. Only closed tickets andtickets with valid values for both these timestamps are used.,3
1,Relevance Observation,DATA_UNDERSTANDING,"Only (number', 'sys_created_at', 'closed_at', 'assignment_group') are needed for this analysis",1
2,Relevance Observation,DATA_UNDERSTANDING,Only ticket activity in the second quarter of 2016 is needed for this analysis,2
3,Data Quality Observation,DATA_UNDERSTANDING,There is inconsistency in the datetime format for both ticket creation and ticket closed times.The parse function from the dateutil library is used to parse times into ISO format and then use that form for the calculation of the time to resolution,4


## Load the Data Representation Observations

In [3]:

load_data_rep_observations(onto2)

,obs_type,intent,finding,finding_seq
0,Feature Engineering Observation,FEATURE_ASSESSMENT,The resolution time attribute is defined. It is calculated as the time difference between closing and creationtimes of the ticket.,1


## Load the Modelling Choice Observations

In [4]:
display(load_modelling_choice_observations(onto2))

""


## Load the Model Selection Observations

In [5]:
load_model_selection_observations(onto2)

,obs_type,intent,finding,finding_seq
0,Model Selection Observation,MODEL_SELECTION,A small group of support groups account for most ticket closures,1
1,Model Selection Observation,MODEL_SELECTION,Many support groups (four from model 1 results) work on longer activity tickets,2


In [6]:
KNOWLEDGE_BASE

'/home/rajiv/programming/kmds_maintainence/KMDS/src/kmds/examples/example_analytics_kb_app_workflow.xml'

In [7]:
from kmds.search import SemanticIndex

idx = SemanticIndex()               # ephemeral (in-memory only)
idx.build(KNOWLEDGE_BASE)        # index all observations

results = idx.search("model selection assumptions", n_results=5)

for r in results:
    print(r["obs_type"], "|", r["finding"])
    print("  distance:", r["distance"])   # cosine distance, lower = more similar

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Model Selection Observation | Many support groups (four from model 1 results) work on longer activity tickets
  distance: 0.828632116317749
Relevance Observation | Only ticket activity in the second quarter of 2016 is needed for this analysis
  distance: 0.8858199119567871
Model Selection Observation | A small group of support groups account for most ticket closures
  distance: 0.9426873922348022
Relevance Observation | Only (number', 'sys_created_at', 'closed_at', 'assignment_group') are needed for this analysis
  distance: 0.9557782411575317
Feature Engineering Observation | The resolution time attribute is defined. It is calculated as the time difference between closing and creationtimes of the ticket.
  distance: 0.9629338383674622


## New Feature 1: Semantic Search Across Observations

This feature builds a semantic vector index from the project knowledge base and lets you ask natural-language queries.

What this gives you:
- You can find relevant findings even when your wording does not exactly match stored text.
- Results include a similarity distance so you can judge relevance (lower is better).

In [8]:
from kmds.search import SemanticIndex

analytics_idx = SemanticIndex()
analytics_idx.build(KNOWLEDGE_BASE)

analytics_query = "What assumptions and trade-offs were made during model selection?"
analytics_semantic_results = analytics_idx.search(analytics_query, n_results=5)

print(f"Query: {analytics_query}")
for i, row in enumerate(analytics_semantic_results, start=1):
    print(f"{i}. {row['obs_type']} | {row['finding']}")
    print(f"   distance={row['distance']:.4f}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: What assumptions and trade-offs were made during model selection?
1. Model Selection Observation | Many support groups (four from model 1 results) work on longer activity tickets
   distance=0.8998
2. Relevance Observation | Only ticket activity in the second quarter of 2016 is needed for this analysis
   distance=0.9196
3. Feature Engineering Observation | The resolution time attribute is defined. It is calculated as the time difference between closing and creationtimes of the ticket.
   distance=0.9331
4. Data Quality Observation | This use case computes the time to resolve a ticket. This is time that has elapsed between the ticket creation and the ticketclosed time stamps. Only closed tickets andtickets with valid values for both these timestamps are used.
   distance=0.9725
5. Relevance Observation | Only (number', 'sys_created_at', 'closed_at', 'assignment_group') are needed for this analysis
   distance=0.9866


## New Feature 2: LLM Search Orchestrator (Template Routing + Fallback)

The orchestrator adds an LLM routing layer on top of search templates.

What this gives you:
- Intent routing: maps a free-form question to the best KMDS template.
- Structured extraction: captures filters (for example, keyword and finding sequence range).
- Safe fallback: if routing fails or no structured records are returned, semantic search is used automatically.

The cell below uses a local demo router function so this notebook works without external API keys.

In [9]:
import json
from kmds.search import SearchOrchestrator


def demo_llm_router_and_synth(prompt: str) -> str:
    # Router response in strict JSON schema expected by the orchestrator.
    if "Return JSON only" in prompt:
        route = {
            "intent_class": "model_selection_search",
            "filters": {
                "keyword": "model",
                "finding_seq_min": 1
            },
            "explanation": "The query asks about model selection assumptions and trade-offs."
        }
        return json.dumps(route)

    # Synthesis response.
    return "Model selection focused on balancing performance and interpretability; see returned records for evidence."


analytics_orchestrator = SearchOrchestrator(
    kb_path=KNOWLEDGE_BASE,
    llm_fn=demo_llm_router_and_synth,
    n_results=5,
)

analytics_orchestrated = analytics_orchestrator.ask(
    "What assumptions and trade-offs were made in selecting the final model?"
)

print("Intent class:", analytics_orchestrated.intent_class)
print("Route explanation:", analytics_orchestrated.route_explanation)
print("Answer:", analytics_orchestrated.answer)
print("Returned records:", len(analytics_orchestrated.results))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Intent class: semantic_search
Route explanation: Fallback – router response could not be parsed.
Answer: Model selection focused on balancing performance and interpretability; see returned records for evidence.
Returned records: 5


## New Feature 3: Natural Language Observation Ingestion

This feature lets you describe a finding in plain English and have KMDS classify it,
extract structured entities, and optionally log it to the knowledge base — without
writing manual ontology code.

Two modes are available inside a notebook:

- **Summary mode**: classify a text statement and get a human-readable description of the matching observation type.
- **Mapping mode**: get the full structured mapping object including the observation family, type, extracted entities, and validation status.

In [ ]:
from kmds.utils.natural_language_observation import summarize_observation_text, map_text_to_observation

# Summary mode: plain-text classification of a finding
text = "Data quality issues were found in the ticket classification field — some records have missing or empty category labels."
print(summarize_observation_text(text))

In [ ]:
# Mapping mode: structured output — family, type, entities, validation
mapping = map_text_to_observation(text)
print(f"Family    : {mapping.workflow_family}")
print(f"Type      : {mapping.observation_type}")
print(f"Intent    : {mapping.intent}")
print(f"Component : {mapping.extracted_entities.affected_component}")
print(f"Valid     : {mapping.validation_passed}")

In [ ]:
# Log mode: convert natural language directly into a KMDS knowledge base entry
# Uncomment the block below to write a new observation to the knowledge base.

# from kmds.utils.natural_language_observation import log_text_as_observation
# result = log_text_as_observation(
#     text=text,
#     workflow_name="ITSM modelling",
#     project_file_path=KNOWLEDGE_BASE,
#     project_mode="update",
# )
# print(result.mapping.observation_type, "logged to", result.project_file)